In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, DateType

# Read from Bronze
df = spark.table("bronze.stock_time_series")

In [0]:
df.printSchema()

In [0]:
# Schema Enforcement

df = df.withColumn(
    "Date",
    F.col("Date").cast(DateType())
)

price_cols = ["Close", "High", "Low", "Open"]

for col_name in price_cols:
    df = df.withColumn(
        col_name,
        F.col(col_name).cast(DoubleType())
    )

df = df.withColumn(
    "Volume",
    F.col("Volume").cast("long")
)


In [0]:
# Data Quality Checks

# Null check
null_count = df.filter(F.col("close").isNull()).count()
assert null_count == 0, f"Null check failed: {null_count} nulls in close"

# Duplicate check
duplicate_count = df.count() - df.dropDuplicates(["Date", "Ticker"]).count()
assert duplicate_count == 0, f"Duplicate check failed: {duplicate_count} duplicates"

# Invalid high/low
invalid_high_low = df.filter(F.col("high") < F.col("low")).count()
assert invalid_high_low == 0, "High < Low detected"

print(f"Quality checks passed — {df.count()} rows")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS silver;

In [0]:
# --- Drop unnecessary columns, add silver metadata ---
df_silver = df.withColumn("processed_at", F.current_timestamp())

# --- Write to Silver ---
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.stock_time_series")